# TPC-H Source Profiling: Finance

This notebook profiles the Databricks TPC-H source dataset before ingestion into the Lakehouse.

The objectives are to:

- understand the structure and volume of the source tables;
- identify nulls and obvious data-quality issues;
- derive Finance-specific validation ranges from the source;
- verify candidate primary/composite keys and foreign-key relationships;
- inspect temporal coverage relevant to Finance analysis;
- verify the consistency of `orders.o_totalprice` with the corresponding line items.

The findings from this notebook are used to define Bronze ingestion checks and Silver-layer validation rules.

> **Scope:** profiling is observational. This notebook does not modify, clean, or filter the source data.

## 1. Source Setup

The assignment uses the TPC-H sample dataset available in Databricks under `samples.tpch`.
All eight source tables are loaded without transformation.

Project-level constants are imported from `src.config` so that source naming remains centralized and portable.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame

from src.config import SOURCE_SCHEMA, TABLES, source_table

print("Source schema:", SOURCE_SCHEMA)
print("Tables:", TABLES)

dfs = {
    table: spark.table(source_table(table))
    for table in TABLES
}

schemas = {
    table: df.schema
    for table, df in dfs.items()
}

## 2. Source Structure and Volume

### 2.1 Table schemas

Schemas are inspected to identify data types and candidate keys before designing Bronze and Silver.

In [0]:
for table in TABLES:
    print(f"\n===== {table.upper()} =====")
    print(schemas[table].treeString())

### 2.2 Row counts

Row counts establish a source baseline. They can later be compared with Bronze counts to verify that raw ingestion has not silently dropped or duplicated rows.

In [0]:
row_counts = [
    (table, df.count())
    for table, df in dfs.items()
]

row_counts_df = spark.createDataFrame(
    row_counts,
    ["table_name", "row_count"],
)

display(row_counts_df.orderBy("table_name"))

## 3. Completeness

Null counts are calculated for every source column. This checks whether fields used in keys, joins, financial calculations, or business segmentation are missing.

In [0]:
def null_counts(
    df: DataFrame,
    columns: list[str],
) -> DataFrame:
    return df.select([
        F.sum(F.col(column).isNull().cast("int")).alias(column)
        for column in columns
    ])


for table in TABLES:
    print(f"\n===== {table.upper()} =====")
    display(
        null_counts(
            dfs[table],
            schemas[table].names,
        )
    )

### Initial observation

The first profiling run found no null values in any source column.

Silver should still retain explicit checks for required keys and Finance-critical fields so that future source changes are detected rather than silently accepted.

## 4. Finance-Specific Numeric Profiling

The Finance requirements ask us to derive valid discount and tax bounds by profiling the source.
Quantity, extended price, and retail price are also inspected because they must be strictly positive.

In [0]:
lineitem = dfs["lineitem"]

finance_profile = lineitem.agg(
    F.min("l_discount").alias("min_discount"),
    F.max("l_discount").alias("max_discount"),
    F.min("l_tax").alias("min_tax"),
    F.max("l_tax").alias("max_tax"),
    F.min("l_quantity").alias("min_quantity"),
    F.max("l_quantity").alias("max_quantity"),
    F.min("l_extendedprice").alias("min_extendedprice"),
    F.max("l_extendedprice").alias("max_extendedprice"),
)

display(finance_profile)

display(
    dfs["part"].agg(
        F.min("p_retailprice").alias("min_retailprice"),
        F.max("p_retailprice").alias("max_retailprice"),
    )
)

### 4.1 Positivity checks

These checks map directly to the Finance validation requirement that quantity, extended price, and retail price be strictly positive.

In [0]:
positivity_checks = {
    "non_positive_quantity":
        lineitem.filter(F.col("l_quantity") <= 0).count(),

    "non_positive_extended_price":
        lineitem.filter(F.col("l_extendedprice") <= 0).count(),

    "non_positive_retail_price":
        dfs["part"].filter(F.col("p_retailprice") <= 0).count(),
}

positivity_checks

### Initial observation

The first profiling run found:

| Field | Minimum | Maximum |
|---|---:|---:|
| `l_discount` | 0.00 | 0.10 |
| `l_tax` | 0.00 | 0.08 |
| `l_quantity` | 1.00 | 50.00 |
| `l_extendedprice` | 900.99 | 104949.50 |
| `p_retailprice` | 900.96 | 2098.99 |

No non-positive values were found for quantity, extended price, or retail price.

These source-derived ranges will be candidates for Silver validation rules.

## 5. Value Domains

Exact values and categories are profiled to detect malformed or unexpected values and to document the domains used later in Finance analytics.

### 5.1 Discount distribution

In [0]:
display(
    lineitem
    .groupBy("l_discount")
    .count()
    .orderBy("l_discount")
)

### 5.2 Tax distribution

In [0]:
display(
    lineitem
    .groupBy("l_tax")
    .count()
    .orderBy("l_tax")
)

### 5.3 Customer market segments

In [0]:
display(
    dfs["customer"]
    .groupBy("c_mktsegment")
    .count()
    .orderBy("c_mktsegment")
)

invalid_market_segments = (
    dfs["customer"]
    .filter(
        F.col("c_mktsegment").isNull()
        | (F.trim(F.col("c_mktsegment")) == "")
    )
    .count()
)

print("Null/blank market segments:", invalid_market_segments)

### Initial observation

The first profiling run showed:

- discount values from `0.00` through `0.10` in increments of `0.01`;
- tax values from `0.00` through `0.08` in increments of `0.01`;
- five market segments: `AUTOMOBILE`, `BUILDING`, `FURNITURE`, `HOUSEHOLD`, and `MACHINERY`;
- no missing/blank market segments.

The five market segments are directly relevant to Finance Question 3 (average order value by market segment).

## 6. Key Integrity

Candidate primary and composite keys are checked for duplicates before the source relational structure is adopted in Silver.

In [0]:
def duplicate_keys(df: DataFrame, *keys: str) -> DataFrame:
    return (
        df.groupBy(*keys)
        .count()
        .filter(F.col("count") > 1)
    )


key_checks = {
    "customer.c_custkey":
        duplicate_keys(dfs["customer"], "c_custkey").count(),

    "orders.o_orderkey":
        duplicate_keys(dfs["orders"], "o_orderkey").count(),

    "supplier.s_suppkey":
        duplicate_keys(dfs["supplier"], "s_suppkey").count(),

    "part.p_partkey":
        duplicate_keys(dfs["part"], "p_partkey").count(),

    "lineitem.(l_orderkey, l_linenumber)":
        duplicate_keys(
            dfs["lineitem"],
            "l_orderkey",
            "l_linenumber",
        ).count(),

    "partsupp.(ps_partkey, ps_suppkey)":
        duplicate_keys(
            dfs["partsupp"],
            "ps_partkey",
            "ps_suppkey",
        ).count(),
}

key_checks

### Initial observation

No duplicate candidate keys were found in the first profiling run.

This supports preserving the normalized TPC-H entity structure in Silver.

## 7. Referential Integrity

Foreign-key relationships are checked for orphan records.

In [0]:
invalid_orders = (
    dfs["orders"]
    .join(
        dfs["customer"],
        dfs["orders"]["o_custkey"] == dfs["customer"]["c_custkey"],
        "left_anti",
    )
)

invalid_lineitems_orders = (
    dfs["lineitem"]
    .join(
        dfs["orders"],
        dfs["lineitem"]["l_orderkey"] == dfs["orders"]["o_orderkey"],
        "left_anti",
    )
)

invalid_lineitems_suppliers = (
    dfs["lineitem"]
    .join(
        dfs["supplier"],
        dfs["lineitem"]["l_suppkey"] == dfs["supplier"]["s_suppkey"],
        "left_anti",
    )
)

invalid_lineitems_parts = (
    dfs["lineitem"]
    .join(
        dfs["part"],
        dfs["lineitem"]["l_partkey"] == dfs["part"]["p_partkey"],
        "left_anti",
    )
)

invalid_customer_nations = (
    dfs["customer"]
    .join(
        dfs["nation"],
        dfs["customer"]["c_nationkey"] == dfs["nation"]["n_nationkey"],
        "left_anti",
    )
)

invalid_supplier_nations = (
    dfs["supplier"]
    .join(
        dfs["nation"],
        dfs["supplier"]["s_nationkey"] == dfs["nation"]["n_nationkey"],
        "left_anti",
    )
)

invalid_nation_regions = (
    dfs["nation"]
    .join(
        dfs["region"],
        dfs["nation"]["n_regionkey"] == dfs["region"]["r_regionkey"],
        "left_anti",
    )
)

invalid_lineitems_partsupp = (
    dfs["lineitem"]
    .join(
        dfs["partsupp"],
        (
            dfs["lineitem"]["l_partkey"]
            == dfs["partsupp"]["ps_partkey"]
        )
        & (
            dfs["lineitem"]["l_suppkey"]
            == dfs["partsupp"]["ps_suppkey"]
        ),
        "left_anti",
    )
)

referential_checks = {
    "orders -> customer":
        invalid_orders.count(),

    "lineitem -> orders":
        invalid_lineitems_orders.count(),

    "lineitem -> supplier":
        invalid_lineitems_suppliers.count(),

    "lineitem -> part":
        invalid_lineitems_parts.count(),

    "customer -> nation":
        invalid_customer_nations.count(),

    "supplier -> nation":
        invalid_supplier_nations.count(),

    "nation -> region":
        invalid_nation_regions.count(),

    "lineitem(part, supplier) -> partsupp":
        invalid_lineitems_partsupp.count(),
}

referential_checks

### Interpretation

A value of `0` for every relationship means no orphan records were found.

The composite `(part, supplier)` relationship between `lineitem` and `partsupp` is particularly important because those two columns together identify a valid supplier offering for a part.

## 8. Temporal Coverage

Order-date coverage is inspected because several Finance questions compare revenue or order value over time.

In [0]:
display(
    dfs["orders"].agg(
        F.min("o_orderdate").alias("min_order_date"),
        F.max("o_orderdate").alias("max_order_date"),
    )
)

display(
    dfs["orders"]
    .withColumn("order_year", F.year("o_orderdate"))
    .groupBy("order_year")
    .count()
    .orderBy("order_year")
)

### Initial observation

Orders span `1992-01-01` through `1998-08-02`.

The years 1992–1997 contain roughly similar full-year order volumes, while 1998 is partial. Therefore:

- 1997 is suitable for the required full-year Finance revenue analysis;
- comparisons involving 1998 must use consistent partial-period boundaries rather than comparing it directly with a complete calendar year.

## 9. Financial Reconciliation

The Finance requirements state that each order total should match the sum of its line-item totals within a fixed tolerance of **$0.01**.


### 9.1 TPC-H reconstruction
TPC-H defines the consistency condition as:

`SUM(trunc(trunc(L_EXTENDEDPRICE × (1 - L_DISCOUNT), 2) × (1 + L_TAX), 2))`

That is, the discounted price is truncated to cents, tax is applied, and the result is truncated to cents again before line totals are summed.

All relevant source fields are positive decimal values, so `floor(x * 100) / 100` implements truncation to two decimal places for this dataset.

In [0]:
def truncate_to_cents(column):
    # TPC-H monetary fields involved here are non-negative.
    # For non-negative values, floor(x * 100) / 100 is truncation to cents.
    return F.floor(column * F.lit(100)) / F.lit(100)


discounted_price = truncate_to_cents(
    F.col("l_extendedprice")
    * (F.lit(1) - F.col("l_discount"))
)

tpch_line_total = truncate_to_cents(
    discounted_price
    * (F.lit(1) + F.col("l_tax"))
)

line_totals_tpch = (
    dfs["lineitem"]
    .withColumn(
        "calculated_line_total",
        tpch_line_total,
    )
    .groupBy("l_orderkey")
    .agg(
        F.sum("calculated_line_total")
        .alias("calculated_order_total")
    )
)

order_reconciliation_tpch = (
    dfs["orders"]
    .select("o_orderkey", "o_totalprice")
    .join(
        line_totals_tpch,
        F.col("o_orderkey") == F.col("l_orderkey"),
        "inner",
    )
    .withColumn(
        "difference",
        F.abs(
            F.col("o_totalprice")
            - F.col("calculated_order_total")
        )
    )
)

display(
    order_reconciliation_tpch
    .orderBy(F.desc("difference"))
    .limit(20)
)

display(
    order_reconciliation_tpch.agg(
        F.max("difference").alias("max_difference"),
        F.avg("difference").alias("avg_difference"),
    )
)

tpch_mismatch_count = (
    order_reconciliation_tpch
    .filter(F.col("difference") > 0.01)
    .count()
)

print("TPC-H mismatches above $0.01:", tpch_mismatch_count)

### 9.2 Reconciliation conclusion

Using the TPC-H-defined calculation semantics, including cent-level
truncation after discount and tax, all orders reconcile with
`o_totalprice` within the required **$0.01 tolerance**.

The number of mismatched orders is **0**, which confirms that the source
financial totals are internally consistent when evaluated according to
the correct TPC-H monetary rules.

This calculation should therefore be used as the basis for the Silver
order-total validation.

## 10. Profiling Conclusions

Based on the initial profiling run:

### Structural quality

- All eight expected TPC-H tables are available.
- No null values were observed in the source.
- Tested candidate primary/composite keys contain no duplicates.
- The original Finance-critical foreign-key checks found no orphan records.

### Finance-specific quality

- `l_discount` ranges from `0.00` to `0.10`.
- `l_tax` ranges from `0.00` to `0.08`.
- `l_quantity`, `l_extendedprice`, and `p_retailprice` are strictly positive in the source.
- Customer market segments contain five observed values:
  `AUTOMOBILE`, `BUILDING`, `FURNITURE`, `HOUSEHOLD`, and `MACHINERY`.

### Temporal coverage

- Orders cover `1992-01-01` through `1998-08-02`.
- 1992–1997 have broadly similar full-year volumes.
- 1998 is partial and should not be compared directly with a complete year without matching the time boundary.

### Financial reconciliation

Order totals were reconstructed using the TPC-H-defined monetary
calculation, including cent-level truncation after discount and tax.

All orders reconcile with `o_totalprice` within the required **$0.01
tolerance**, with **0 mismatches** observed.

This confirms that the source financial totals are internally
consistent and that the same reconciliation rule can be enforced in
the Silver layer.

## 11. Implications for Silver

The profiling results suggest that the Silver layer should:

1. preserve the normalized TPC-H entity structure;
2. retain explicit non-null checks for keys and Finance-critical fields;
3. validate the source-derived ranges:
   - `0.00 <= l_discount <= 0.10`
   - `0.00 <= l_tax <= 0.08`;
4. require:
   - `l_quantity > 0`
   - `l_extendedprice > 0`
   - `p_retailprice > 0`;
5. enforce or explicitly validate candidate primary/composite keys;
6. validate the documented foreign-key relationships;
7. validate `c_mktsegment` against the five observed source values;
8. reconcile `o_totalprice` using the TPC-H-defined truncation semantics and the assignment's `$0.01` tolerance.